# Independent m6A-Atlas Evaluation Notebook

用于在 m6A-Atlas 独立测试集文件夹上批量预测，并输出每个 CSV 的预测结果和 metrics。对应正式脚本：

```text
evaluate_independent_atlas.py
```


## Step 0. 参数设置


In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "lamar_m6a").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

print(PROJECT_ROOT)

/picb/rnasys2/lidaoqian/LAMAR-M6A


In [10]:
from pathlib import Path

MODE = "full"  # full / mature
LENGTH = 801
INPUT_FOLDER = Path("data/independent_test_sets/atlas_a/human_full")
PATTERN = "*.csv"
CHECKPOINT_PATH = Path("outputs/finetune/full/a549-101-bs16-acc4/checkpoint-3100")
RUN_NAME = "a549_full_801"
OUTPUTS_DIR = Path("outputs")

TOKENIZER_PATH = Path("tokenizer/single_nucleotide")
MODEL_CONFIG = Path("configs/config_150M.json")
MODEL_CLASS = "scripts.sequence_classification_patch:EsmForSequenceClassification"
MODEL_MAX_LENGTH = LENGTH + 2
INFER_BATCH_SIZE = 512
DEVICE = None


## Step 1. 导入函数


In [11]:
from argparse import Namespace
from lamar_m6a.evaluation.evaluate_independent_atlas import find_model_state, load_model, predict_csv
from transformers import AutoTokenizer
import pandas as pd


## Step 2. 检查输入文件夹和 checkpoint


In [12]:
input_files = sorted(INPUT_FOLDER.glob(PATTERN))
print("n files:", len(input_files))
for path in input_files[:5]:
    print(path)

model_state_path = find_model_state(CHECKPOINT_PATH)
print("checkpoint:", model_state_path)


n files: 38
data/independent_test_sets/atlas_a/human_full/A549_m6A-CLIP-seq_merged.csv
data/independent_test_sets/atlas_a/human_full/A549_m6A-seq_with_improved_protocol_merged.csv
data/independent_test_sets/atlas_a/human_full/Brain_m6A-REF-seq_merged.csv
data/independent_test_sets/atlas_a/human_full/CD8T_m6A-CLIP-seq_merged.csv
data/independent_test_sets/atlas_a/human_full/CM2005.1_miCLIP__merged.csv
checkpoint: outputs/finetune/full/a549-101-bs16-acc4/checkpoint-3100/model.safetensors


## Step 3. 加载模型和 tokenizer


In [9]:
args = Namespace(
    mode=MODE,
    length=LENGTH,
    input_folder=INPUT_FOLDER,
    pattern=PATTERN,
    checkpoint_path=CHECKPOINT_PATH,
    run_name=RUN_NAME,
    outputs_dir=OUTPUTS_DIR,
    tokenizer_path=TOKENIZER_PATH,
    model_config=MODEL_CONFIG,
    model_class=MODEL_CLASS,
    model_max_length=MODEL_MAX_LENGTH,
    padding_side="left",
    use_fast_tokenizer=False,
    num_labels=2,
    token_dropout=False,
    positional_embedding_type="rotary",
    hidden_size=768,
    intermediate_size=3072,
    num_attention_heads=12,
    num_hidden_layers=12,
    head_type="Linear",
    kernel_sizes=None,
    ocs=None,
    strict_load=True,
    infer_batch_size=INFER_BATCH_SIZE,
    device=DEVICE,
)

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH, model_max_length=MODEL_MAX_LENGTH, padding_side="left", use_fast=False)
model = load_model(args, model_state_path, tokenizer)
model


EsmForSequenceClassification(
  (esm): EsmModel(
    (embeddings): EsmEmbeddings(
      (word_embeddings): Embedding(10, 768, padding_idx=1)
      (dropout): Dropout(p=0.0, inplace=False)
      (position_embeddings): Embedding(1026, 768, padding_idx=1)
    )
    (encoder): EsmEncoder(
      (layer): ModuleList(
        (0-11): 12 x EsmLayer(
          (attention): EsmAttention(
            (self): EsmSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
              (rotary_embeddings): RotaryEmbedding()
            )
            (output): EsmSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
            (LayerNorm): LayerNorm((768,), eps=1e-0

## Step 4. 批量预测 m6A-Atlas 文件夹


In [ ]:
summary_rows = []
prediction_dir = OUTPUTS_DIR / "predictions" / "independent_atlas" / MODE / RUN_NAME
for csv_path in input_files:
    output_path = prediction_dir / f"{csv_path.stem}_pred.csv"
    metrics = predict_csv(args, model, tokenizer, csv_path, output_path)
    print(csv_path.name, metrics)
    if metrics is not None:
        row = {"mode": MODE, "run_name": RUN_NAME, "length": LENGTH, "input_csv": str(csv_path), "prediction_csv": str(output_path)}
        row.update(metrics)
        summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_path = OUTPUTS_DIR / "metrics" / "independent_atlas" / MODE / f"{RUN_NAME}_{LENGTH}_metrics.csv"
summary_path.parent.mkdir(parents=True, exist_ok=True)
summary_df.to_csv(summary_path, index=False)
summary_df
